In [27]:
##### Evaluate RF models 
# calculates model performance metrics 

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [28]:
##### SET-UP

# Get the current working directory
cd = Path.cwd().parent.parent 

# import country averages 
country_avg = pd.read_csv(f'{cd}/Data/Clean/Predictors/Vectors/country_intensities.csv')

# set directories
RESULTS_DIR = Path(f"{cd}/Results/RF_models_final")

test = pd.read_parquet(RESULTS_DIR /'labor_rf_spatial_CV/predictions.parquet')

In [29]:
##### Add country averages to predictions
test = test.merge(country_avg[['PROJ_ID', 'country_labor_intensity_jobs_per_tonne']], on='PROJ_ID', how='left')

##### Convert country intensity to correct scale and log1p
test['log_country_labor_intensity_jobs_per_million_tonne'] = np.log1p(test['country_labor_intensity_jobs_per_tonne'] * 1e6)

#### Estimate actual predicted intensity (not relative)
test['predicted_log_labor_intensity_jobs_per_million_tonne'] = test['prediction'] + test['log_country_labor_intensity_jobs_per_million_tonne']


In [26]:
test

Index(['PROJ_ID', 'country_ID',
       'rtv_log_labor_intensity_jobs_per_million_tonne', 'prediction', 'fold',
       'split', 'country_labor_intensity_jobs_per_tonne',
       'log_country_labor_intensity_jobs_per_tonne'],
      dtype='object')

In [14]:

def compute_metrics_by_fold(df, target_col, pred_col):
    rows = []
    for fold, g in df.groupby("fold"):
        y_true = g[target_col]
        y_pred = g[pred_col]
        resid  = y_true - y_pred

        rows.append({
            "fold":      fold,
            "n":         len(g),
            "R2":        r2_score(y_true, y_pred),
            "RMSE":      np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAE":       mean_absolute_error(y_true, y_pred),
            "Bias":      resid.mean(),
            "Pearson_r": y_true.corr(y_pred),
        })

    metrics_df = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)

    # summary rows across folds (unweighted — averaged, not pooled)
    metric_cols = ["R2", "RMSE", "MAE", "Bias", "Pearson_r"]
    mean_row = {"fold": "mean", "n": metrics_df["n"].sum()}
    var_row  = {"fold": "variance", "n": np.nan}
    for col in metric_cols:
        mean_row[col] = metrics_df[col].mean()
        var_row[col]  = metrics_df[col].var()  

    summary = pd.DataFrame([mean_row, var_row])
    return pd.concat([metrics_df, summary], ignore_index=True)

target_col = "rtv_log_labor_intensity_jobs_per_million_tonne" 

test_df  = test[test["split"] == "test"]
train_df = test[test["split"] == "train"]

test_metrics  = compute_metrics_by_fold(test_df,  target_col, 'prediction')
train_metrics = compute_metrics_by_fold(train_df, target_col, 'prediction')

print("TEST")
print(test_metrics.to_string(index=False))

print("\nTRAIN")
print(train_metrics.to_string(index=False))

TEST
    fold      n        R2     RMSE      MAE      Bias  Pearson_r
  fold_1 1019.0  0.510566 1.075400 0.866948  0.314286   0.800528
  fold_2 1071.0  0.658988 0.762938 0.607847  0.017389   0.813307
  fold_3  548.0 -0.076289 0.696047 0.529080 -0.285271   0.497095
  fold_4  546.0  0.534472 0.917121 0.710815 -0.153639   0.740210
  fold_5  547.0  0.539893 0.834541 0.629489  0.031505   0.737123
    mean 3731.0  0.433526 0.857209 0.668836 -0.015146   0.717653
variance    NaN  0.084546 0.021643 0.016453  0.050977   0.016388

TRAIN
    fold       n       R2     RMSE      MAE     Bias  Pearson_r
  fold_1  2712.0 0.825100 0.504282 0.371943 0.002872   0.918058
  fold_2  2660.0 0.821177 0.574096 0.430722 0.017985   0.919683
  fold_3  3183.0 0.854324 0.544434 0.410581 0.019128   0.933176
  fold_4  3185.0 0.855396 0.509714 0.383927 0.013158   0.934933
  fold_5  3184.0 0.855700 0.519857 0.390266 0.018895   0.934557
    mean 14924.0 0.842340 0.530477 0.397488 0.014407   0.928082
variance     NaN 0.0